# Phase 08 — Overall Impression Signals

**Status:** Complete  
**Workflow:** Notebook-first training documentation  
**Purpose:** Define grounded overallImpression signals that can be rendered by an API wrapper without inventing unsupported claims.

This notebook is the Phase 8 source of truth. It writes `reports/phase_08_overall_impression_signals.json` with the input signal policy, deterministic template plan, fallback policy, wrapper boundary, quality checks, and current implementation decision.


## Contract boundary

`overallImpression` is a public API string. The model/core may only emit structured evidence-backed impression signals and a recommended template ID. The API wrapper owns final prose, tone, localization, and any product-specific presentation.

The impression must summarize only observed job-fit strengths, observed job-fit gaps, ATS risks, and confidence limitations. It must not create new skills, seniority, personality, eligibility, hiring likelihood, or job-detail claims.


## Shared setup

### Purpose
Load prior job-fit, ATS, language policy, and generated OpenAPI contract evidence, then define helpers used by every Phase 8 step.

### Required input
Repository root with `GAP_MODEL_TRAINING.md`, `reports/phase_03_normalization_feature_design.json`, `reports/phase_06_jobfit_training_experiments.json`, `reports/phase_07_ats_friendliness_scoring.json`, and `references/docs/generated/openapi.json`.

### Action
Read inherited signal contracts, confirm `overallImpression` is exposed as a string in the public API, and prepare a stable report path for Phase 8 decisions.

### Expected output
Reusable variables for public schema constraints, language values, model-owned signal names, ATS issue keys, and final report writing.

### Verification
Fail fast if any required input is missing. Confirm the public `overallImpression` field remains a string and that language values still include ID, EN, MIXED, and UNKNOWN.


In [5]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "GAP_MODEL_TRAINING.md").exists() and (candidate / "training").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from notebook runtime.")


ROOT = find_repo_root(Path.cwd())
REPORTS = ROOT / "reports"
OPENAPI_PATH = ROOT / "references" / "docs" / "generated" / "openapi.json"
PHASE3_PATH = REPORTS / "phase_03_normalization_feature_design.json"
PHASE6_PATH = REPORTS / "phase_06_jobfit_training_experiments.json"
PHASE7_PATH = REPORTS / "phase_07_ats_friendliness_scoring.json"
REPORT_PATH = REPORTS / "phase_08_overall_impression_signals.json"


def read_json(path: Path) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Required Phase 8 input is missing: {path.relative_to(ROOT)}")
    return json.loads(path.read_text())


openapi = read_json(OPENAPI_PATH)
phase3 = read_json(PHASE3_PATH)
phase6 = read_json(PHASE6_PATH)
phase7 = read_json(PHASE7_PATH)

analysis_props = openapi["components"]["schemas"]["CvAnalysis"]["properties"]["analysisResult"]["properties"]
overall_schema = analysis_props["overallImpression"]
assert overall_schema == {"type": "string"}

language_values = set(phase3["language_normalization_policy"]["allowed_values"])
assert {"ID", "EN", "MIXED", "UNKNOWN"}.issubset(language_values)

jobfit_fields = {field["field"] for field in phase6["output_signal_contract"]["model_owned_fields"]}
ats_fields = {field["field"] for field in phase7["output_contract"]["model_owned_fields"]}
ats_issue_keys = sorted({key for issue in phase7["issue_taxonomy"] for key in issue["issue_keys"]})

setup_summary = {
    "public_overall_impression_schema": overall_schema,
    "language_values": sorted(language_values),
    "jobfit_model_owned_fields": sorted(jobfit_fields),
    "ats_model_owned_fields": sorted(ats_fields),
    "ats_issue_key_count": len(ats_issue_keys),
    "inherited_blockers": sorted(set(phase6.get("blocked_until_later_phases", []) + phase7.get("blocked_until_later_phases", []))),
}
setup_summary


{'public_overall_impression_schema': {'type': 'string'},
 'language_values': ['EN', 'ID', 'MIXED', 'UNKNOWN'],
 'jobfit_model_owned_fields': ['confidenceNotes',
  'matchedSkills',
  'missingSignals',
  'missingSkills',
  'score',
  'summarySignals'],
 'ats_model_owned_fields': ['componentScores',
  'confidenceNotes',
  'detectedIssues',
  'score'],
 'ats_issue_key_count': 22,
 'inherited_blockers': ['Classifier training is blocked until benchmark labels exist with enough examples per issue family.',
  'Complex JobFitAlignment training must wait for a materialized balanced pair dataset with high-fit validation/test coverage.',
  'Production ATS score semantics require Phase 10 calibration evidence.',
  'Production score semantics must wait for Phase 10 calibration.',
  'Promotion must wait for human validation labels or equivalent trusted validation evidence.',
  'Ranking-objective promotion must wait for backend-like candidate sets and relevance labels.',
  'Release claims are blocked 

## Step 8.1 — Input signal policy

### Purpose
List which signals can be used for an impression and which fields must never be inferred without evidence.

### Required input
Use Phase 6 job-fit output signals, Phase 7 ATS output signals, Phase 3 language normalization policy, target job evidence, and parser/data quality evidence. Do not use wrapper-generated copy as model input.

### Action
Create allowed and forbidden signal catalogs with source evidence, safe usage, and non-inference rules. Any signal that lacks source evidence must be excluded from impression rendering.

### Expected output
A machine-readable input signal policy for future core output, wrapper mapping, and review checks.

### Verification
Every allowed job-fit and ATS field maps to prior phase contracts. Forbidden fields include unsupported skills, seniority, protected attributes, hiring likelihood, and backend-owned fields.


In [6]:
allowed_input_signals = [
    {
        "signal": "jobFitAlignment.score",
        "source_contract": "phase_06.output_signal_contract.score",
        "safe_use": "Select score band language only after Phase 10 calibration; before calibration use as draft evidence with confidence note.",
        "required_evidence": ["core_score", "model_version", "score_range_0_100"],
        "must_not_infer": ["hiring_likelihood", "interview_probability", "candidate_quality_beyond_target_job"],
    },
    {
        "signal": "jobFitAlignment.summarySignals",
        "source_contract": "phase_06.output_signal_contract.summarySignals",
        "safe_use": "Mention observed strengths such as role match, strong skill overlap, experience match, or requirement coverage.",
        "required_evidence": ["signal_key", "source_evidence_key"],
        "must_not_infer": ["unstated_skill", "personality_trait", "culture_fit"],
    },
    {
        "signal": "jobFitAlignment.missingSignals",
        "source_contract": "phase_06.output_signal_contract.missingSignals",
        "safe_use": "Mention observed gaps such as missing required skill, experience gap, low requirement coverage, or unknown language.",
        "required_evidence": ["signal_key", "source_evidence_key"],
        "must_not_infer": ["candidate_inability", "career_seniority_downgrade", "unverified_experience_gap"],
    },
    {
        "signal": "jobFitAlignment.matchedSkills",
        "source_contract": "phase_06.output_signal_contract.matchedSkills",
        "safe_use": "Name matched skills only when the normalized candidate skill and job skill intersection exists.",
        "required_evidence": ["candidate_skill_source", "job_skill_source", "normalization_rule"],
        "must_not_infer": ["skill_proficiency", "years_of_skill_experience", "certification"],
    },
    {
        "signal": "jobFitAlignment.missingSkills",
        "source_contract": "phase_06.output_signal_contract.missingSkills",
        "safe_use": "Name missing required job skills only when they appear in job requirements and are absent from normalized candidate evidence.",
        "required_evidence": ["job_required_skill_source", "candidate_skill_absence_check"],
        "must_not_infer": ["candidate_cannot_learn_skill", "job_disqualification"],
    },
    {
        "signal": "atsFriendliness.score",
        "source_contract": "phase_07.output_contract.score",
        "safe_use": "Mention CV/ATS quality band only after Phase 10 calibration; before calibration use as draft evidence with caveat.",
        "required_evidence": ["ats_core_score", "component_scores", "score_range_0_100"],
        "must_not_infer": ["job_fit", "candidate_competence", "applicant_tracking_system_outcome"],
    },
    {
        "signal": "atsFriendliness.detectedIssues",
        "source_contract": "phase_07.output_contract.detectedIssues",
        "safe_use": "Mention ATS risks only from stable issue keys and severity.",
        "required_evidence": ["issueKey", "family", "severity", "sourceEvidenceKeys"],
        "must_not_infer": ["raw_contact_value", "document_authenticity", "employment_verification"],
    },
    {
        "signal": "targetJobEvidence",
        "source_contract": "job title, role family, required skills, requirements, and description text available to model core",
        "safe_use": "Mention target role or requirement only when present in the request/job record.",
        "required_evidence": ["job_id_or_request_id", "source_field"],
        "must_not_infer": ["job_availability", "company_preference", "salary_fit", "location_fit_without_policy"],
    },
    {
        "signal": "languageSignal",
        "source_contract": "phase_03.language_normalization_policy",
        "safe_use": "Choose or flag localization mode; UNKNOWN must trigger fallback or wrapper locale decision.",
        "required_evidence": ["normalized_language", "language_source_or_marker_counts"],
        "must_not_infer": ["fluency", "nationality", "preferred_work_language"],
    },
]

forbidden_inferences = [
    {"category": "skills", "rule": "Do not mention a skill unless it is in matchedSkills, missingSkills, or target job requirements with source evidence."},
    {"category": "seniority", "rule": "Do not claim junior, mid, senior, lead, or manager unless target role/seniority or normalized experience evidence explicitly supports it."},
    {"category": "employment_history", "rule": "Do not infer tenure, employer quality, gaps, promotions, or verified work history from CV text unless explicit evidence exists."},
    {"category": "protected_attributes", "rule": "Never infer age, gender, religion, ethnicity, marital status, health, disability, or nationality."},
    {"category": "hiring_outcome", "rule": "Never claim ATS pass/fail, interview chance, hiring probability, or recruiter preference."},
    {"category": "backend_owned_fields", "rule": "Do not use topActionables, sectionReviews, hydrated job details, auth, persistence, or wrapper summaries as model evidence."},
    {"category": "private_contact_values", "rule": "Do not render raw email, phone, URL, address, or other PII in impression text."},
]

allowed_signal_names = {item["signal"] for item in allowed_input_signals}
assert {"jobFitAlignment.score", "jobFitAlignment.matchedSkills", "atsFriendliness.detectedIssues", "languageSignal"}.issubset(allowed_signal_names)
assert all("source" in key or "evidence" in key for item in allowed_input_signals for key in item["required_evidence"])
assert {"skills", "seniority", "protected_attributes", "hiring_outcome", "backend_owned_fields"}.issubset({item["category"] for item in forbidden_inferences})
allowed_input_signals


## Step 8.2 — Grounded summary design

### Purpose
Document deterministic summary templates that mention only observed strengths, gaps, and ATS risks.

### Required input
Use allowed signals from Step 8.1 plus score bands, matched skills, missing skills, ATS issue keys, language signal, and confidence notes.

### Action
Define template IDs, triggering conditions, required slots, evidence rules, and prohibited slot content. Templates must be deterministic and safe before wrapper copywriting.

### Expected output
A template catalog that an API wrapper can render into `overallImpression` without inventing unsupported claims.

### Verification
Every template includes required evidence keys and avoids unsupported seniority, skill, hiring outcome, and PII claims.


### Retired executable reference

This cell was intentionally retired during Phase 27 notebook hygiene. Durable Phase 8 evidence lives in `reports/phase_08_overall_impression_signals.json`. The original code is preserved below for audit context only.

```python
summary_templates = [
    {
        "template_id": "fit_strength_with_minor_gaps_v1",
        "condition": "jobfit_band in high_or_upper_medium and at least one matched_skill or strength signal exists",
        "required_slots": ["target_role_or_job_context", "strength_evidence"],
        "optional_slots": ["top_gap_evidence", "ats_risk_evidence", "confidence_caveat"],
        "template_en": "Strong alignment for {target_role_or_job_context} based on {strength_evidence}; review {top_gap_evidence} and {ats_risk_evidence} before applying.",
        "template_id_locale": "Kesesuaian kuat untuk {target_role_or_job_context} berdasarkan {strength_evidence}; tinjau {top_gap_evidence} dan {ats_risk_evidence} sebelum melamar.",
        "evidence_rule": "strength_evidence must come from summarySignals or matchedSkills; gap/risk clauses omitted when evidence is absent.",
    },
    {
        "template_id": "mixed_fit_actionable_gaps_v1",
        "condition": "jobfit_band in medium and at least one strength plus one gap exists",
        "required_slots": ["target_role_or_job_context", "strength_evidence", "top_gap_evidence"],
        "optional_slots": ["ats_risk_evidence", "confidence_caveat"],
        "template_en": "Moderate alignment for {target_role_or_job_context}: {strength_evidence} is supported, while {top_gap_evidence} needs attention.",
        "template_id_locale": "Kesesuaian sedang untuk {target_role_or_job_context}: {strength_evidence} sudah terlihat, sementara {top_gap_evidence} perlu diperbaiki.",
        "evidence_rule": "Both strength and gap evidence must have distinct sourceEvidenceKeys.",
    },
    {
        "template_id": "low_fit_or_missing_requirements_v1",
        "condition": "jobfit_band in low or multiple required gaps are present",
        "required_slots": ["target_role_or_job_context", "top_gap_evidence"],
        "optional_slots": ["available_strength_evidence", "ats_risk_evidence", "confidence_caveat"],
        "template_en": "Current evidence shows limited alignment for {target_role_or_job_context}, mainly because {top_gap_evidence}.",
        "template_id_locale": "Bukti saat ini menunjukkan kesesuaian terbatas untuk {target_role_or_job_context}, terutama karena {top_gap_evidence}.",
        "evidence_rule": "Do not say the candidate is unqualified; describe only observed missing requirements or low coverage.",
    },
    {
        "template_id": "ats_risk_dominant_v1",
        "condition": "critical or high ATS issue exists, regardless of job-fit score",
        "required_slots": ["ats_risk_evidence"],
        "optional_slots": ["strength_evidence", "top_gap_evidence", "target_role_or_job_context"],
        "template_en": "The main limitation is CV readability or structure: {ats_risk_evidence}. Use job-fit signals only after the CV can be parsed reliably.",
        "template_id_locale": "Batasan utama ada pada keterbacaan atau struktur CV: {ats_risk_evidence}. Gunakan sinyal job-fit setelah CV dapat diparse dengan andal.",
        "evidence_rule": "ATS risk evidence must come from detectedIssues with severity high or critical.",
    },
    {
        "template_id": "confidence_limited_v1",
        "condition": "low confidence note, unknown language, missing job target, or incomplete parser evidence exists",
        "required_slots": ["confidence_caveat"],
        "optional_slots": ["available_strength_evidence", "top_gap_evidence", "ats_risk_evidence"],
        "template_en": "The impression is limited because {confidence_caveat}. Only the observed signals should be used for decisions.",
        "template_id_locale": "Impresi ini terbatas karena {confidence_caveat}. Gunakan hanya sinyal yang benar-benar teramati untuk keputusan.",
        "evidence_rule": "Fallback or caveat must be rendered when required role, language, parser, or confidence evidence is missing.",
    },
]

slot_policy = {
    "target_role_or_job_context": "Use explicit target role, job title, or generic 'target role/job' fallback; never infer seniority from title text alone.",
    "strength_evidence": "One or two items from summarySignals or matchedSkills with source evidence.",
    "available_strength_evidence": "Optional observed strength; omit if none exists.",
    "top_gap_evidence": "One or two items from missingSignals or missingSkills with source evidence.",
    "ats_risk_evidence": "One critical/high ATS issue or concise aggregate from detectedIssues; no raw PII.",
    "confidence_caveat": "Data-quality or model-confidence note from confidenceNotes, parser status, unknown language, or missing job target.",
}

prohibited_template_claims = [
    "will pass ATS",
    "will get interview",
    "is senior-level",
    "has unstated skill",
    "is a strong culture fit",
    "verified employment history",
    "raw contact value",
]

assert len(summary_templates) >= 5
assert all(template["template_id"].endswith("_v1") for template in summary_templates)
assert all(template["required_slots"] for template in summary_templates)
assert all("evidence" in template["evidence_rule"] for template in summary_templates)
summary_templates
```


## Step 8.3 — Fallback handling

### Purpose
Define safe summaries for empty CV parse, missing job target, unknown language, and low-confidence model output.

### Required input
Use parser status, text coverage, target job evidence, normalized language, job-fit confidence notes, ATS detected issues, and model confidence notes.

### Action
Define fallback triggers, safe output intent, allowed evidence, blocked claims, and wrapper rendering guidance. Fallbacks must prefer honest uncertainty over invented summary quality.

### Expected output
A fallback policy that covers empty CV parse, missing job target, unknown language, low-confidence output, and partial signal availability.

### Verification
Every required fallback case exists and each fallback blocks unsupported skill, seniority, and hiring outcome claims.


### Retired executable reference

This cell was intentionally retired during Phase 27 notebook hygiene. Durable Phase 8 evidence lives in `reports/phase_08_overall_impression_signals.json`. The original code is preserved below for audit context only.

```python
fallback_policy = [
    {
        "fallback_id": "empty_cv_parse",
        "trigger": "empty_text issue, extracted_character_count below minimum, or parser_status indicates failed parse",
        "safe_summary_intent": "Explain that the CV could not be read well enough for a reliable overall impression.",
        "allowed_evidence": ["parser_status", "extracted_character_count", "ats_issue_key.empty_text", "ats_issue_key.ocr_required"],
        "blocked_claims": ["job_fit_strength", "missing_skill", "seniority", "skill_proficiency", "hiring_outcome"],
        "wrapper_guidance": "Render a short parseability caveat and ask for a readable CV; do not summarize job fit.",
    },
    {
        "fallback_id": "missing_job_target",
        "trigger": "no job title, role target, requirements, or candidate job context is supplied",
        "safe_summary_intent": "Summarize only CV/ATS signals and state that job-specific fit cannot be assessed.",
        "allowed_evidence": ["atsFriendliness.detectedIssues", "atsFriendliness.score", "general_cv_text_coverage"],
        "blocked_claims": ["target_role_alignment", "missing_required_skill", "requirement_coverage", "job_specific_seniority"],
        "wrapper_guidance": "Use generic 'target role' wording only when product UX requires a placeholder; otherwise state missing job target.",
    },
    {
        "fallback_id": "unknown_language",
        "trigger": "normalized language is UNKNOWN or language evidence conflicts across sources",
        "safe_summary_intent": "Keep wording language-neutral or use user/request locale while flagging language uncertainty.",
        "allowed_evidence": ["languageSignal.UNKNOWN", "marker_counts", "trusted_source_language_absent"],
        "blocked_claims": ["language_fluency", "nationality", "preferred_work_language", "localized_skill_translation_without_alias"],
        "wrapper_guidance": "Prefer configured UI locale; do not translate skill names unless alias table supports it.",
    },
    {
        "fallback_id": "low_confidence_model_output",
        "trigger": "confidenceNotes contain low_text_coverage, out_of_distribution_role, unknown_experience, unstable_score, or calibration_missing",
        "safe_summary_intent": "Show limited impression and list only strongest observed evidence with caveat.",
        "allowed_evidence": ["confidenceNotes", "observed matchedSkills", "observed detectedIssues"],
        "blocked_claims": ["precise_score_semantics", "unqualified_claim", "strong_recommendation", "career_level_claim"],
        "wrapper_guidance": "Use confidence_limited_v1 template and avoid decisive language.",
    },
    {
        "fallback_id": "no_strength_or_gap_signal",
        "trigger": "score exists but summarySignals, missingSignals, matchedSkills, missingSkills, and detectedIssues are empty",
        "safe_summary_intent": "State that there is not enough explainable evidence to produce an impression.",
        "allowed_evidence": ["score_presence", "empty_signal_sets"],
        "blocked_claims": ["strength", "gap", "ATS risk", "seniority", "hiring_outcome"],
        "wrapper_guidance": "Render a minimal uncertainty message and log quality check failure for review.",
    },
]

required_fallbacks = {"empty_cv_parse", "missing_job_target", "unknown_language", "low_confidence_model_output"}
assert required_fallbacks.issubset({fallback["fallback_id"] for fallback in fallback_policy})
assert all("hiring_outcome" in fallback["blocked_claims"] or fallback["fallback_id"] == "unknown_language" for fallback in fallback_policy)
fallback_policy
```


## Step 8.4 — Wrapper boundary

### Purpose
Explain how the model/core output can be converted by the API wrapper into the OpenAPI `overallImpression` string.

### Required input
Use Step 8.1 allowed signals, Step 8.2 templates, Step 8.3 fallback policy, Phase 6 job-fit core contract, Phase 7 ATS core contract, and generated OpenAPI schema.

### Action
Define a core-only structured output contract and a wrapper mapping into the public string. Keep product copy, localization, truncation, and API envelope concerns outside model training.

### Expected output
A clear boundary document that lets the API wrapper render `analysisResult.overallImpression` without making unsupported claims.

### Verification
The public OpenAPI field remains a string, while model/core output remains structured and evidence-backed. Wrapper-owned fields are listed as out of scope.


### Retired executable reference

This cell was intentionally retired during Phase 27 notebook hygiene. Durable Phase 8 evidence lives in `reports/phase_08_overall_impression_signals.json`. The original code is preserved below for audit context only.

```python
core_output_contract = {
    "schema_version": "overall-impression-core-v1",
    "model_owned_fields": [
        {
            "field": "templateId",
            "type": "string",
            "range": sorted(template["template_id"] for template in summary_templates),
            "grounding_rule": "Chosen deterministically from allowed signals, fallback triggers, and quality checks.",
        },
        {
            "field": "strengthSignals",
            "type": "array[object]",
            "range": "0-3 observed strengths",
            "grounding_rule": "Each item must map to summarySignals or matchedSkills with sourceEvidenceKeys.",
            "item_shape": {"label": "display-safe evidence label", "sourceEvidenceKeys": "array[string]"},
        },
        {
            "field": "gapSignals",
            "type": "array[object]",
            "range": "0-3 observed gaps",
            "grounding_rule": "Each item must map to missingSignals or missingSkills with sourceEvidenceKeys.",
            "item_shape": {"label": "display-safe evidence label", "sourceEvidenceKeys": "array[string]"},
        },
        {
            "field": "atsRiskSignals",
            "type": "array[object]",
            "range": "0-3 observed ATS risks",
            "grounding_rule": "Each item must map to Phase 7 detectedIssues and must not expose raw PII.",
            "item_shape": {"issueKey": "stable Phase 7 issue key", "severity": "low | medium | high | critical", "sourceEvidenceKeys": "array[string]"},
        },
        {
            "field": "fallbackReason",
            "type": "string | null",
            "range": sorted(fallback["fallback_id"] for fallback in fallback_policy),
            "grounding_rule": "Set when fallback trigger fires; wrapper must render uncertainty instead of normal template.",
        },
        {
            "field": "languageMode",
            "type": "ID | EN | MIXED | UNKNOWN",
            "range": sorted(language_values),
            "grounding_rule": "Derived from Phase 3 language normalization; UNKNOWN delegates locale choice to wrapper with caveat.",
        },
        {
            "field": "confidenceNotes",
            "type": "array[string]",
            "range": "0-5 diagnostic notes",
            "grounding_rule": "Only data quality, parser, calibration, language, or out-of-distribution notes.",
        },
        {
            "field": "evidenceLedger",
            "type": "array[object]",
            "range": "one row per rendered slot",
            "grounding_rule": "Every rendered slot must identify source path and sourceEvidenceKeys.",
            "item_shape": {"slot": "template slot name", "sourcePath": "core signal path", "sourceEvidenceKeys": "array[string]"},
        },
    ],
    "public_api_mapping": {
        "analysisResult.overallImpression": "API wrapper renders one concise string from templateId, evidence-backed slots, fallbackReason, languageMode, and confidenceNotes.",
        "analysisResult.jobFitAlignment.summary": "Rendered separately by wrapper from job-fit signals; not an input to overall impression core.",
        "analysisResult.atsFriendliness.summary": "Rendered separately by wrapper from ATS signals; not an input to overall impression core.",
    },
    "not_model_owned_fields": [
        "final prose tone",
        "localized product copy",
        "topActionables",
        "sectionReviews",
        "jobRecommendations",
        "hydrated job details",
        "auth",
        "persistence",
        "request validation",
        "OpenAPI envelope",
    ],
    "wrapper_requirements": [
        "Render only slots with evidenceLedger entries.",
        "Drop optional clauses whose evidence is missing.",
        "Prefer fallback copy when fallbackReason is not null.",
        "Keep the public value a string as required by OpenAPI.",
        "Do not read generated jobFitAlignment.summary or atsFriendliness.summary as evidence.",
    ],
}

assert overall_schema == {"type": "string"}
assert core_output_contract["schema_version"] == "overall-impression-core-v1"
assert any(field["field"] == "evidenceLedger" for field in core_output_contract["model_owned_fields"])
assert "topActionables" in core_output_contract["not_model_owned_fields"]
core_output_contract
```


## Step 8.5 — Quality checks

### Purpose
Define checks that prevent hallucinated skills, unsupported seniority claims, and language mismatch.

### Required input
Use the core output contract, evidence ledger, allowed/forbidden signal policy, summary templates, fallback policy, normalized skills, normalized language, target job evidence, and wrapper-rendered string.

### Action
Create validation checks with fail conditions, severity, and remediation rules. These checks should run before any rendered `overallImpression` is accepted.

### Expected output
A quality check catalog and promotion gate for hallucination prevention, seniority safety, language consistency, evidence coverage, and fallback enforcement.

### Verification
The quality catalog includes explicit checks for hallucinated skills, unsupported seniority claims, and language mismatch. Blocking failures prevent rendering or force fallback.


### Retired executable reference

This cell was intentionally retired during Phase 27 notebook hygiene. Durable Phase 8 evidence lives in `reports/phase_08_overall_impression_signals.json`. The original code is preserved below for audit context only.

```python
quality_checks = [
    {
        "check_id": "rendered_slot_has_evidence",
        "severity": "blocking",
        "prevents": "unsupported_claim",
        "condition": "Every rendered slot has an evidenceLedger entry with sourcePath and non-empty sourceEvidenceKeys.",
        "fail_action": "Drop the slot; if required slot missing, render fallback confidence_limited_v1.",
    },
    {
        "check_id": "no_hallucinated_skills",
        "severity": "blocking",
        "prevents": "hallucinated_skills",
        "condition": "Every skill token in the rendered string appears in matchedSkills, missingSkills, target job required skills, or approved alias table.",
        "fail_action": "Remove skill mention and log evidence mismatch; block release if repeated.",
    },
    {
        "check_id": "seniority_claim_supported",
        "severity": "blocking",
        "prevents": "unsupported_seniority_claim",
        "condition": "Any junior/mid/senior/lead/manager wording has explicit source evidence from target role, normalized experience, or job requirement.",
        "fail_action": "Replace with neutral target-role wording.",
    },
    {
        "check_id": "language_mode_consistent",
        "severity": "blocking",
        "prevents": "language_mismatch",
        "condition": "Rendered locale matches languageMode or wrapper UI locale when languageMode is UNKNOWN; skill aliases are not translated without alias evidence.",
        "fail_action": "Use wrapper locale fallback and preserve original skill labels.",
    },
    {
        "check_id": "fallback_trigger_enforced",
        "severity": "blocking",
        "prevents": "unsafe_confident_summary",
        "condition": "If fallbackReason is set, only the matching fallback template may render.",
        "fail_action": "Replace normal template with fallback output.",
    },
    {
        "check_id": "no_hiring_or_ats_outcome_claim",
        "severity": "blocking",
        "prevents": "unsupported_outcome_claim",
        "condition": "Rendered string contains no pass/fail, interview chance, recruiter preference, or hiring probability language.",
        "fail_action": "Reject rendered string and log prohibited phrase.",
    },
    {
        "check_id": "no_pii_rendering",
        "severity": "blocking",
        "prevents": "privacy_leak",
        "condition": "Rendered string contains no raw email, phone, URL, address, or identifier values from CV/contact evidence.",
        "fail_action": "Remove PII-bearing phrase and force manual review.",
    },
    {
        "check_id": "ats_jobfit_boundary_respected",
        "severity": "blocking",
        "prevents": "signal_boundary_mixup",
        "condition": "ATS evidence describes CV quality only; job-fit evidence describes alignment only.",
        "fail_action": "Split or remove mixed claim.",
    },
    {
        "check_id": "length_and_clause_limit",
        "severity": "warning",
        "prevents": "overly_broad_summary",
        "condition": "Rendered impression stays concise and uses no more than one strength clause, one gap clause, and one risk/caveat clause.",
        "fail_action": "Trim lowest-priority optional clause.",
    },
]

promotion_gate = {
    "current_decision": "GO for grounded signal and wrapper design; NO-GO for production copy claims until Phase 10 calibration and wrapper implementation checks exist.",
    "blocking_checks_must_pass": [check["check_id"] for check in quality_checks if check["severity"] == "blocking"],
    "manual_review_required_for": [
        "fallback rates above expected threshold",
        "unknown language or low text coverage clusters",
        "any detected PII rendering",
        "any unsupported seniority or skill claim",
    ],
    "phase10_dependency": "Score-band wording and confidence semantics must be calibrated before release claims are finalized.",
}

check_ids = {check["check_id"] for check in quality_checks}
assert {"no_hallucinated_skills", "seniority_claim_supported", "language_mode_consistent"}.issubset(check_ids)
assert all(check["severity"] in {"blocking", "warning"} for check in quality_checks)
assert "no_hallucinated_skills" in promotion_gate["blocking_checks_must_pass"]

phase8_report = {
    "schema_version": "phase-08-overall-impression-signals-v1",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "inputs": {
        "openapi_json": str(OPENAPI_PATH.relative_to(ROOT)),
        "phase3_report": str(PHASE3_PATH.relative_to(ROOT)),
        "phase6_report": str(PHASE6_PATH.relative_to(ROOT)),
        "phase7_report": str(PHASE7_PATH.relative_to(ROOT)),
    },
    "setup_summary": setup_summary,
    "allowed_input_signals": allowed_input_signals,
    "forbidden_inferences": forbidden_inferences,
    "summary_templates": summary_templates,
    "slot_policy": slot_policy,
    "prohibited_template_claims": prohibited_template_claims,
    "fallback_policy": fallback_policy,
    "core_output_contract": core_output_contract,
    "quality_checks": quality_checks,
    "promotion_gate": promotion_gate,
    "blocked_until_later_phases": [
        "Production score-band wording requires Phase 10 calibration evidence.",
        "API wrapper implementation must enforce evidence ledger, fallback, and quality checks before release.",
        "Manual review is required for PII, unsupported skill, unsupported seniority, or language mismatch failures.",
    ],
    "acceptance": {
        "impression_rules_are_grounded_in_model_signals": True,
        "fallback_policy_exists": True,
        "wrapper_boundary_is_clear": True,
    },
}

REPORTS.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(phase8_report, indent=2, sort_keys=True) + "\n")
{
    "report_path": str(REPORT_PATH.relative_to(ROOT)),
    "template_count": len(summary_templates),
    "fallback_count": len(fallback_policy),
    "quality_check_count": len(quality_checks),
    "acceptance": phase8_report["acceptance"],
}
```


## Acceptance criteria

- [x] Impression rules are grounded in model signals.
- [x] Fallback policy exists.
- [x] Wrapper boundary is clear.


## Phase notes

Phase 8 is complete for notebook-first contract design. Production wording remains blocked until Phase 10 calibrates score-band semantics and the API wrapper enforces evidence-ledger and quality checks.
